In [15]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

In [10]:
df=pd.read_csv('all_tickets_processed_improved_v3.csv')

In [11]:
df

,Document,Topic_group
0,connection with icon icon dear please setup ic...,Hardware
1,work experience user work experience user hi w...,Access
2,requesting for meeting requesting meeting hi p...,Hardware
3,reset passwords for external accounts re expir...,Access
4,mail verification warning hi has got attached ...,Miscellaneous
...,...,...
47832,git space for a project issues with adding use...,Access
47833,error sent july error hi guys can you help out...,Miscellaneous
47834,connection issues sent tuesday july connection...,Hardware
47835,error cube reports sent tuesday july error hel...,HR Support


In [ ]:

# function to apply to all the rows of teh data frame 
def global_clean(text):
    # 1.convert everything to lowercase
    text = str(text).lower()
    
    # 2.remove days of the week and months which are not relevant to the technical content of the tickets
    text = re.sub(r'\b(monday|tuesday|wednesday|thursday|friday|saturday|sunday)\b', ' ', text)
    text = re.sub(r'\b(january|february|march|april|may|june|july|august|september|october|november|december)\b', ' ', text)
    
    # 3.remove common email boilerplate that carries no technical meaning
    text = re.sub(r'\b(hi|hello|dear|thanks|regards|please|guys|issue|problem|help)\b', ' ', text)
    
    # 4.remove all punctuation, special characters, and numbers
    text = re.sub(r'[^a-z\s]', ' ', text)
    
    # 5.compress multiple spaces down to a single space and strip edges
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text


text_column = 'Document' 

df['step1_cleaned'] = df[text_column].apply(global_clean)



In [13]:
df

,Document,Topic_group,step1_cleaned
0,connection with icon icon dear please setup ic...,Hardware,connection with icon icon setup icon per icon ...
1,work experience user work experience user hi w...,Access,work experience user work experience user work...
2,requesting for meeting requesting meeting hi p...,Hardware,requesting for meeting requesting meeting help...
3,reset passwords for external accounts re expir...,Access,reset passwords for external accounts re expir...
4,mail verification warning hi has got attached ...,Miscellaneous,mail verification warning has got attached add...
...,...,...,...
47832,git space for a project issues with adding use...,Access,git space for a project issues with adding use...
47833,error sent july error hi guys can you help out...,Miscellaneous,error sent error can you help out with error a...
47834,connection issues sent tuesday july connection...,Hardware,connection issues sent connection issues have ...
47835,error cube reports sent tuesday july error hel...,HR Support,error cube reports sent error have received er...


In [ ]:
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True) 
nltk.download('wordnet', quiet=True)
nltk.download('stopwords', quiet=True)

# 2.initialize the Lemmatizer and load English stop words
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))



def lemmatize_text(text):
    # Tokenize: Break the string into a list of words
    words = word_tokenize(str(text))
    
    # Filter and Lemmatize
    processed_words = [
        lemmatizer.lemmatize(word) 
        for word in words 
        if word not in stop_words and len(word) > 2 # Drop lingering 1-2 letter artifacts
    ]
    
    # Rejoin the words back into a single string for TF-IDF later
    return ' '.join(processed_words)



# Apply this to the column we generated in Step 1
df['step2_lemmatized'] = df['step1_cleaned'].apply(lemmatize_text)



Tokenizing and Lemmatizing (this may take a minute for 47,000 rows)...
                                       step1_cleaned  \
0  connection with icon icon setup icon per icon ...   
1  work experience user work experience user work...   
2  requesting for meeting requesting meeting help...   
3  reset passwords for external accounts re expir...   
4  mail verification warning has got attached add...   

                                    step2_lemmatized  
0  connection icon icon setup icon per icon engin...  
1  work experience user work experience user work...  
2  requesting meeting requesting meeting help fol...  
3  reset password external account expire day ask...  
4  mail verification warning got attached address...  


In [17]:
df

,Document,Topic_group,step1_cleaned,step2_lemmatized
0,connection with icon icon dear please setup ic...,Hardware,connection with icon icon setup icon per icon ...,connection icon icon setup icon per icon engin...
1,work experience user work experience user hi w...,Access,work experience user work experience user work...,work experience user work experience user work...
2,requesting for meeting requesting meeting hi p...,Hardware,requesting for meeting requesting meeting help...,requesting meeting requesting meeting help fol...
3,reset passwords for external accounts re expir...,Access,reset passwords for external accounts re expir...,reset password external account expire day ask...
4,mail verification warning hi has got attached ...,Miscellaneous,mail verification warning has got attached add...,mail verification warning got attached address...
...,...,...,...,...
47832,git space for a project issues with adding use...,Access,git space for a project issues with adding use...,git space project issue adding user sent git s...
47833,error sent july error hi guys can you help out...,Miscellaneous,error sent error can you help out with error a...,error sent error help error appearing want sub...
47834,connection issues sent tuesday july connection...,Hardware,connection issues sent connection issues have ...,connection issue sent connection issue connect...
47835,error cube reports sent tuesday july error hel...,HR Support,error cube reports sent error have received er...,error cube report sent error received error tr...


In [ ]:
from sklearn.model_selection import train_test_split

# 1.define your features (X) and target labels (y)
x = df['step2_lemmatized']

# y is the target column containing the ticket categories
y = df['Topic_group'] 

# 2.perform the split
x_train, x_test, y_train, y_test = train_test_split(
    x, 
    y, 
    test_size=0.2, 
    random_state=69,
    stratify=y  #crucial for imbalanced data portfolios!
)

# 3. Verify the split sizes 
# we know is it imbalanced from the documentation on kaggle 
print(f"Training features shape: {x_train.shape}")
print(f"Testing features shape: {x_test.shape}")
print(f"Training labels distribution:\n{y_train.value_counts(normalize=True) * 100}")

Training features shape: (38269,)
Testing features shape: (9568,)
Training labels distribution:
Topic_group
Hardware                 28.464292
HR Support               22.817424
Access                   14.894562
Miscellaneous            14.758682
Storage                   5.806266
Purchase                  5.150383
Internal Project          4.429172
Administrative rights     3.679218
Name: proportion, dtype: float64


In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Initialize the Vectorizer
# max_features=5000 limits the vocabulary to the top 5000 most mathematically significant words, saving RAM.
# ngram_range=(1, 2) allows the model to learn single words ("server") AND two-word phrases ("server down").
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))

# 2. Fit AND Transform the Training Data
# The .fit_transform() method looks at X_train, builds the 5000-word dictionary, 
# calculates the IDF weights, and outputs the numerical matrix.
print("Vectorizing training data...")
X_train_tfidf = tfidf.fit_transform(X_train)

# 3. ONLY Transform the Testing Data
# CRITICAL: We use .transform() here, NOT .fit_transform(). 
# This forces the test data to be scored using ONLY the vocabulary and weights learned from the training data.
print("Vectorizing testing data...")
X_test_tfidf = tfidf.transform(X_test)

# 4. Verify the transformation
print(f"\nTraining Matrix Shape: {X_train_tfidf.shape}")
print(f"Testing Matrix Shape:  {X_test_tfidf.shape}")

Vectorizing training data...
Vectorizing testing data...

Training Matrix Shape: (38269, 5000)
Testing Matrix Shape:  (9568, 5000)


In [20]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# Initialize the "Brain"
# max_iter=1000 gives it enough time to learn without timing out. 
# class_weight='balanced' forces it to pay attention to small departments.
model = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')

print("Cell 1 complete: Model initialized and ready to learn!")

Cell 1 complete: Model initialized and ready to learn!


In [21]:
# 1. Train the model using the training data
print("Training the Logistic Regression model (this might take a few seconds)...")

# The .fit() command looks at the TF-IDF scores and calculates the routing rules.
model.fit(X_train_tfidf, y_train)

print("Cell 2 complete: The model has finished learning the routing patterns!")

Training the Logistic Regression model (this might take a few seconds)...
Cell 2 complete: The model has finished learning the routing patterns!


In [22]:
# 1. The Final Exam: Make predictions on the hidden Test Set
print("Taking the final exam...")
predictions = model.predict(X_test_tfidf)

# 2. Grade the Exam
print("\n--- Model Report Card ---")
print(f"Overall Accuracy: {accuracy_score(y_test, predictions) * 100:.2f}%\n")

# The classification report breaks down the score for EVERY single department
print(classification_report(y_test, predictions, zero_division=0))

Taking the final exam...

--- Model Report Card ---
Overall Accuracy: 84.47%

                       precision    recall  f1-score   support

               Access       0.90      0.88      0.89      1425
Administrative rights       0.63      0.86      0.73       352
           HR Support       0.89      0.84      0.86      2183
             Hardware       0.86      0.78      0.82      2724
     Internal Project       0.79      0.94      0.86       424
        Miscellaneous       0.79      0.86      0.82      1412
             Purchase       0.90      0.91      0.91       493
              Storage       0.81      0.93      0.86       555

             accuracy                           0.84      9568
            macro avg       0.82      0.87      0.84      9568
         weighted avg       0.85      0.84      0.85      9568



In [23]:
from sklearn.svm import LinearSVC

# 1. Initialize the SVM model
# We still use class_weight='balanced' to protect the rare departments.
# max_iter=2000 gives it plenty of time to find the perfect mathematical boundaries.
print("Initializing the Support Vector Machine (SVM)...")
model = LinearSVC(random_state=42, class_weight='balanced', max_iter=2000)

# 2. Train the model using the training data
print("Training the SVM model (this might take a few seconds)...")
model.fit(X_train_tfidf, y_train)

print("Cell 2 complete: The SVM has finished learning the routing patterns!")

Initializing the Support Vector Machine (SVM)...
Training the SVM model (this might take a few seconds)...
Cell 2 complete: The SVM has finished learning the routing patterns!


In [24]:
from sklearn.metrics import classification_report, accuracy_score

# 1. The Final Exam: Make predictions on the hidden Test Set
print("Taking the final exam with the SVM...")
predictions = model.predict(X_test_tfidf)

# 2. Grade the Exam
print("\n--- SVM Model Report Card ---")
print(f"Overall Accuracy: {accuracy_score(y_test, predictions) * 100:.2f}%\n")

# The classification report breaks down the score for EVERY single department
print(classification_report(y_test, predictions, zero_division=0))

Taking the final exam with the SVM...

--- SVM Model Report Card ---
Overall Accuracy: 84.59%

                       precision    recall  f1-score   support

               Access       0.89      0.90      0.90      1425
Administrative rights       0.71      0.78      0.74       352
           HR Support       0.87      0.85      0.86      2183
             Hardware       0.84      0.80      0.82      2724
     Internal Project       0.81      0.92      0.86       424
        Miscellaneous       0.80      0.84      0.82      1412
             Purchase       0.89      0.90      0.89       493
              Storage       0.85      0.91      0.88       555

             accuracy                           0.85      9568
            macro avg       0.83      0.86      0.85      9568
         weighted avg       0.85      0.85      0.85      9568

